In [1]:
# Optimized VAE imputation without histogram gradient boosting
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, optimizers, regularizers, initializers, callbacks
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
from scipy import special
import warnings
import random
import os
import gc
import time
import hashlib

warnings.filterwarnings("ignore")

# Check the TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

# ==================== Enhanced reproducibility settings ====================
SEED = 42

def set_deterministic_mode():
    """Enable deterministic mode."""
    # Python random seed
    random.seed(SEED)
    
    # Python hash seed
    os.environ['PYTHONHASHSEED'] = str(SEED)
    
    # NumPy random seed
    np.random.seed(SEED)
    
    # TensorFlow random seed
    tf.random.set_seed(SEED)
    
    # TensorFlow determinism settings
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    
    # Use single-threaded TensorFlow execution for reproducibility
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
    
    print("Deterministic mode enabled")

# Initialize deterministic mode
set_deterministic_mode()

# Configure the GPU for deterministic execution or disable it
USE_GPU = False  # Set to False to avoid potentially nondeterministic GPU operations

if USE_GPU:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            # Limit GPU memory growth
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            # Enable deterministic GPU operations
            tf.config.experimental.enable_op_determinism()
            print(f"Found {len(gpus)} GPU devices. Deterministic mode is enabled.")
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
    else:
        print("No GPU found. Running on the CPU.")
else:
    # Disable the GPU
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    print("GPU disabled. Running on the CPU for reproducibility.")

# Clear the session
tf.keras.backend.clear_session()
gc.collect()

# Create a random number generator for sampling
rng = np.random.RandomState(SEED)
tf_random = tf.random.Generator.from_seed(SEED)

# Implement the inverse Box-Cox transform
def inv_boxcox(y, lambda_param):
    """Apply the inverse Box-Cox transform."""
    if lambda_param == 0:
        return np.exp(y)
    else:
        # Avoid invalid results from negative values
        return np.power(lambda_param * y + 1, 1 / lambda_param)

# 1. Data loading and preprocessing
def load_and_preprocess_data():
    """Load the data and perform initial preprocessing."""
    # Use a fixed random seed
    local_rng = np.random.RandomState(SEED)
    
    data = pd.read_csv("./Rainfall.csv", low_memory=True)
    print(f"Original data shape: {data.shape}")
    
    # Keep a copy of the original data
    original_data = data.copy()
    
    # Drop nonnumeric columns in a deterministic order
    cols_to_drop = []
    for col in sorted(data.columns):  # Sort to ensure consistent order
        try:
            pd.to_numeric(data[col].dropna().iloc[:10])
        except:
            cols_to_drop.append(col)
    
    data = data.drop(columns=sorted(cols_to_drop))  # Sort to ensure consistent order
    
    # Drop ID columns in a deterministic order
    cols_sorted = sorted(data.columns[:3])  # Sort to ensure consistent order
    for col in cols_sorted:
        if data[col].dtype in ['int64', 'float64']:
            non_null = data[col].dropna()
            if len(non_null) > 1:
                diffs = non_null.diff().dropna()
                if (diffs == 1).mean() > 0.9:
                    data = data.drop(columns=[col])
    
    # Drop the L(m) column
    if 'L(m)' in data.columns:
        data = data.drop(columns=['L(m)'])
    
    # Drop columns with more than 30% missing values
    missing_ratio = data.isnull().sum() / len(data)
    cols_to_drop = sorted(missing_ratio[missing_ratio > 0.30].index)  # Sort to ensure consistency
    data = data.drop(columns=cols_to_drop)
    
    # Convert data types
    for col in sorted(data.columns):  # Sort to ensure consistent order
        data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')
    
    # Reorder the columns consistently
    data = data[sorted(data.columns)]
    
    print(f"Final data shape: {data.shape}")
    print(f"Features: {list(data.columns)}")
    
    return data, original_data

# 2. Outlier treatment with gentler handling for As
def optimized_outlier_handling(df):
    """Apply feature-specific outlier treatment."""
    df_clean = df.copy()
    transform_info = {}
    
    # Sort column names for consistent processing order
    columns_sorted = sorted(df_clean.columns)
    
    for column in columns_sorted:
        mask = df_clean[column].notna()
        values = df_clean.loc[mask, column].values
        
        if len(values) == 0:
            continue
        
        # Use the existing treatment for V(m³)
        if 'V' in column:
            print(f"{column}: applying a quantile transform to a highly skewed feature")
            
            qt = QuantileTransformer(n_quantiles=min(1000, len(values)), 
                                    output_distribution='normal', 
                                    random_state=SEED,
                                    subsample=100000)
            values_transformed = qt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {'type': 'quantile', 'transformer': qt}
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            
        elif 'H' in column:
            # Keep the simplified treatment for H(m)
            print(f"{column}: applying simplified robust preprocessing")
            
            # Clip outliers using a wide range
            q1, q3 = np.percentile(values, [10, 90])
            iqr = q3 - q1
            lower = q1 - 2.0 * iqr
            upper = q3 + 2.0 * iqr
            values_clipped = np.clip(values, lower, upper)
            
            # Apply basic robust scaling
            median = np.median(values_clipped)
            mad = np.median(np.abs(values_clipped - median))
            if mad > 0:
                values_standardized = (values_clipped - median) / (mad * 1.4826)
            else:
                values_standardized = values_clipped - median
            
            transform_info[column] = {
                'type': 'h_robust', 
                'clip_lower': lower,
                'clip_upper': upper,
                'median': median,
                'mad': mad
            }
            df_clean[column] = df_clean[column].astype(values_standardized.dtype, copy=False)
            df_clean.loc[mask, column] = values_standardized
            
        elif 'As' in column:
            # Apply gentler treatment to As to preserve more information
            print(f"{column}: applying a gentler power transform")
            
            # Use wider outlier bounds
            q1, q3 = np.percentile(values, [5, 95])
            iqr = q3 - q1
            lower = q1 - 2.5 * iqr  # Widen the clipping range
            upper = q3 + 2.5 * iqr
            values = np.clip(values, lower, upper)
            
            # Use PowerTransformer for greater numerical stability
            pt = PowerTransformer(method='yeo-johnson', standardize=True, copy=True)
            values_transformed = pt.fit_transform(values.reshape(-1, 1)).flatten()
            
            transform_info[column] = {
                'type': 'power', 
                'transformer': pt,
                'clip_lower': lower,
                'clip_upper': upper
            }
            df_clean[column] = df_clean[column].astype(values_transformed.dtype, copy=False)
            df_clean.loc[mask, column] = values_transformed
            
        elif 'θ' in column:
            # Use stronger regularization for the angle feature
            print(f"{column}: applying stronger regularization")
            
            # Use tight outlier bounds
            lower, upper = np.percentile(values, [3, 97])
            values = np.clip(values, lower, upper)
            
            # Add noise to reduce overfitting
            noise = np.random.RandomState(SEED).normal(0, 0.025 * np.std(values), len(values))
            values = values + noise
            
            # Robust scaling
            median = np.median(values)
            mad = np.median(np.abs(values - median))
            if mad > 0:
                values = (values - median) / (mad * 1.4826)
            else:
                values = values - median
            
            transform_info[column] = {
                'type': 'theta_regularized', 
                'median': median,
                'mad': mad,
                'lower': lower,
                'upper': upper
            }
            df_clean[column] = df_clean[column].astype(values.dtype, copy=False)
            df_clean.loc[mask, column] = values
            
        else:
            # Use robust preprocessing for Wsm(m)
            print(f"{column}: applying robust outlier treatment")
            
            # Use an IQR multiplier of 1.5
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            values = np.clip(values, lower, upper)
            transform_info[column] = {'type': 'clip', 'shift': 0}
            df_clean[column] = df_clean[column].astype(values.dtype, copy=False)
            df_clean.loc[mask, column] = values
    
    return df_clean, transform_info

# 3. Feature-specific scaling with separate handling for As
def optimized_normalization(data):
    """Apply optimized feature scaling."""
    data_normalized = pd.DataFrame(index=data.index, columns=data.columns)
    scalers = {}
    
    # Sort column names for consistent processing order
    columns_sorted = sorted(data.columns)
    
    for column in columns_sorted:
        mask = data[column].notna()
        values = data[column][mask].values.reshape(-1, 1)
        
        if len(values) > 0:
            # Choose the scaler by feature
            if 'V' in column:
                # Use StandardScaler for V
                scaler = StandardScaler()
            elif 'H' in column:
                # Keep RobustScaler for H(m)
                scaler = RobustScaler(quantile_range=(10, 90))
            elif 'As' in column:
                # Use StandardScaler for As to preserve more information
                scaler = StandardScaler()
            elif 'θ' in column:
                # Use RobustScaler with tighter bounds for θ to increase regularization
                scaler = RobustScaler(quantile_range=(25, 75))
            else:
                # Use RobustScaler for the remaining features
                scaler = RobustScaler(quantile_range=(5, 95))
            
            scaler.fit(values)
            scaled = scaler.transform(values).flatten()
            
            # Map values to a suitable range
            min_val, max_val = scaled.min(), scaled.max()
            if max_val - min_val > 1e-8:
                if 'H' in column:
                    # Use a wide range for H(m)
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.9 + 0.05
                elif 'As' in column:
                    # Use the full range for As
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.95 + 0.025
                elif 'θ' in column:
                    # Regularize θ with a narrow range
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.5 + 0.25
                else:
                    scaled = (scaled - min_val) / (max_val - min_val) * 0.8 + 0.1
            else:
                scaled = np.ones_like(scaled) * 0.5
            
            data_normalized.loc[mask, column] = scaled.astype(np.float32)
            scalers[column] = {'scaler': scaler, 'min': min_val, 'max': max_val}
    
    return data_normalized, scalers

# Inverse scaling
def inverse_transform_array(data_normalized_array, scalers, transform_info, original_columns, original_index=None):
    """Convert the scaled NumPy array to a DataFrame in the original units."""
    n_samples = data_normalized_array.shape[0]
    
    if original_index is None:
        original_index = range(n_samples)
    
    data_original = pd.DataFrame(index=original_index, columns=original_columns)
    
    for i, column in enumerate(original_columns):
        if column not in scalers:
            data_original[column] = data_normalized_array[:, i]
            continue
            
        values = data_normalized_array[:, i].copy()
        
        # Reverse the scaling
        scaler_info = scalers[column]
        min_val = scaler_info['min']
        max_val = scaler_info['max']
        
        # Recover the values from the mapped range
        if 'H' in column:
            values = (values - 0.05) / 0.9 * (max_val - min_val) + min_val
        elif 'As' in column:
            values = (values - 0.025) / 0.95 * (max_val - min_val) + min_val
        elif 'θ' in column:
            values = (values - 0.25) / 0.5 * (max_val - min_val) + min_val
        else:
            values = (values - 0.1) / 0.8 * (max_val - min_val) + min_val
        
        # Invert the scaler
        values = scaler_info['scaler'].inverse_transform(values.reshape(-1, 1)).flatten()
        
        # Invert the transform
        if column in transform_info:
            trans_type = transform_info[column]['type']
            
            if trans_type == 'quantile':
                # Invert the quantile transform
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'h_robust':
                # Simplified inverse transform for H(m)
                median = transform_info[column]['median']
                mad = transform_info[column]['mad']
                if mad > 0:
                    values = values * (mad * 1.4826) + median
                else:
                    values = values + median
            elif trans_type == 'power':
                # Invert the power transform for As
                transformer = transform_info[column]['transformer']
                values = transformer.inverse_transform(values.reshape(-1, 1)).flatten()
            elif trans_type == 'theta_regularized':
                # Inverse transform for θ
                median = transform_info[column]['median']
                mad = transform_info[column]['mad']
                if mad > 0:
                    values = values * (mad * 1.4826) + median
                else:
                    values = values + median
        
        data_original[column] = values
    
    return data_original

# 4. Optimized VAE with feature-specific weights
class OptimizedVAE:
    def __init__(self, data_dim, latent_dim=14, learning_rate=0.0004, seed=42, column_names=None):
        """Optimized VAE model."""
        self.data_dim = data_dim
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.seed = seed
        self.column_names = column_names
        
        # Create the feature-weight mask
        self.feature_weights = np.ones(data_dim, dtype=np.float32)
        self.regularization_weights = np.ones(data_dim, dtype=np.float32)
        
        if column_names is not None:
            for i, col in enumerate(column_names):
                if 'H' in col:
                    self.feature_weights[i] = 4.0  # Keep a high weight for H(m)
                    self.regularization_weights[i] = 0.4
                elif 'As' in col:
                    self.feature_weights[i] = 3.0  # Increase the weight for As
                    self.regularization_weights[i] = 0.5  # Reduce regularization
                elif 'θ' in col:
                    self.feature_weights[i] = 0.5  # Decrease the weight for θ
                    self.regularization_weights[i] = 4.0  # Increase regularization
                elif 'V' in col:
                    self.feature_weights[i] = 1.5
                    self.regularization_weights[i] = 0.8
        
        # Convert to a TensorFlow constant
        self.feature_weights_tf = tf.constant(self.feature_weights.reshape(1, -1), dtype=tf.float32)
        self.regularization_weights_tf = tf.constant(self.regularization_weights.reshape(1, -1), dtype=tf.float32)
        
        # Set the seed
        tf.random.set_seed(seed)
        
        # Use seeded initializers
        self.initializer = initializers.GlorotUniform(seed=seed)
        
        # Build the model
        self.encoder = self._build_encoder()
        self.decoder = self._build_decoder()
        
        # Optimizer
        self.optimizer = optimizers.Adam(learning_rate=learning_rate)
        
        self.best_weights = None
        self.best_val_loss = float('inf')
        
        # Seeded random number generator
        self.random_generator = tf.random.Generator.from_seed(seed)
    
    def _build_encoder(self):
        """Build the encoder."""
        inputs = layers.Input(shape=(self.data_dim,))
        
        x = layers.Dense(128, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.1, seed=self.seed)(x)
        
        x = layers.Dense(64, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.08, seed=self.seed+1)(x)
        
        x = layers.Dense(32, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0002))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        z_mean = layers.Dense(self.latent_dim, kernel_initializer=self.initializer)(x)
        z_log_var = layers.Dense(self.latent_dim, kernel_initializer=self.initializer)(x)
        
        # Limit the range of log_var
        z_log_var = layers.Lambda(lambda x: tf.clip_by_value(x, -3, 3))(z_log_var)
        
        return keras.Model(inputs, [z_mean, z_log_var])
    
    def _build_decoder(self):
        """Build the decoder."""
        inputs = layers.Input(shape=(self.latent_dim,))
        
        x = layers.Dense(32, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0002))(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Dense(64, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.08, seed=self.seed+2)(x)
        
        x = layers.Dense(128, kernel_initializer=self.initializer,
                        kernel_regularizer=regularizers.l2(0.0003))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(0.1, seed=self.seed+3)(x)
        
        outputs = layers.Dense(self.data_dim, activation='sigmoid', 
                              kernel_initializer=self.initializer)(x)
        
        return keras.Model(inputs, outputs)
    
    def _sample(self, z_mean, z_log_var, training=True):
        """Sample using the reparameterization trick."""
        if training:
            batch = tf.shape(z_mean)[0]
            dim = tf.shape(z_mean)[1]
            epsilon = self.random_generator.normal(shape=(batch, dim))
            return z_mean + tf.exp(0.5 * z_log_var) * epsilon * 0.06
        else:
            return z_mean
    
    @tf.function
    def train_step(self, x_batch, mask_batch, kl_weight=0.0003):
        """Run one training step."""
        with tf.GradientTape() as tape:
            # Encode
            z_mean, z_log_var = self.encoder(x_batch, training=True)
            z = self._sample(z_mean, z_log_var, training=True)
            
            # Decode
            reconstructed = self.decoder(z, training=True)
            
            # Calculate reconstruction loss
            mse = tf.square(x_batch - reconstructed) * mask_batch
            
            # Apply the feature weights
            batch_size = tf.shape(x_batch)[0]
            weights = tf.tile(self.feature_weights_tf, [batch_size, 1])
            
            weighted_mse = mse * weights
            weighted_mask = mask_batch * weights
            
            reconstruction_loss = tf.reduce_sum(weighted_mse) / (tf.reduce_sum(weighted_mask) + 1e-8)
            
            # KL loss
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )
            
            # Total loss
            total_loss = reconstruction_loss + kl_weight * kl_loss
        
        # Update gradients
        variables = self.encoder.trainable_variables + self.decoder.trainable_variables
        gradients = tape.gradient(total_loss, variables)
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        self.optimizer.apply_gradients(zip(gradients, variables))
        
        return total_loss
    
    def fit(self, data, mask, validation_split=0.2, epochs=900, batch_size=64):
        """Train the model."""
        # Split the data
        n_samples = len(data)
        indices = np.arange(n_samples)
        local_rng = np.random.RandomState(self.seed)
        local_rng.shuffle(indices)
        
        split_point = int(n_samples * (1 - validation_split))
        train_indices = indices[:split_point]
        val_indices = indices[split_point:]
        
        x_train = data[train_indices]
        x_val = data[val_indices]
        mask_train = mask[train_indices]
        mask_val = mask[val_indices]
        
        print(f"Training set: {len(x_train)} samples")
        print(f"Validation set: {len(x_val)} samples")
        
        # Early-stopping settings
        patience = 70
        patience_counter = 0
        
        # Learning-rate schedule
        lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
            self.learning_rate,
            decay_steps=len(x_train) // batch_size * epochs,
            alpha=0.03
        )
        self.optimizer.learning_rate = lr_schedule
        
        for epoch in range(epochs):
            # Adjust the KL weight during training
            kl_weight = min(0.0008, 0.00003 * (epoch / 100))
            
            # Train
            local_rng = np.random.RandomState(self.seed + epoch)
            train_indices_shuffled = local_rng.permutation(len(x_train))
            train_loss = 0
            n_batches = 0
            
            for i in range(0, len(x_train), batch_size):
                batch_indices = train_indices_shuffled[i:i+batch_size]
                loss = self.train_step(
                    x_train[batch_indices],
                    mask_train[batch_indices],
                    kl_weight
                )
                train_loss += loss
                n_batches += 1
            
            train_loss /= n_batches
            
            # Validation
            if epoch % 10 == 0:
                val_loss = 0
                n_val_batches = 0
                
                for i in range(0, len(x_val), batch_size):
                    z_mean, z_log_var = self.encoder(x_val[i:i+batch_size], training=False)
                    z = z_mean
                    reconstructed = self.decoder(z, training=False)
                    
                    mse = tf.square(x_val[i:i+batch_size] - reconstructed) * mask_val[i:i+batch_size]
                    val_loss += tf.reduce_sum(mse) / tf.reduce_sum(mask_val[i:i+batch_size])
                    n_val_batches += 1
                
                val_loss /= n_val_batches
                
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    self.best_weights = {
                        'encoder': self.encoder.get_weights(),
                        'decoder': self.decoder.get_weights()
                    }
                    patience_counter = 0
                else:
                    patience_counter += 10
                
                if epoch % 50 == 0:
                    print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, Val Loss={val_loss:.6f}")
                
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}")
                    break
        
        # Restore the best weights
        if self.best_weights:
            self.encoder.set_weights(self.best_weights['encoder'])
            self.decoder.set_weights(self.best_weights['decoder'])
    
    def impute(self, data, mask):
        """Impute missing values."""
        imputed = np.zeros_like(data)
        batch_size = 128
        
        for i in range(0, len(data), batch_size):
            batch_data = data[i:i+batch_size]
            batch_mask = mask[i:i+batch_size]
            
            z_mean, _ = self.encoder(batch_data, training=False)
            pred = self.decoder(z_mean, training=False).numpy()
            
            imputed[i:i+batch_size] = batch_data * batch_mask + pred * (1 - batch_mask)
        
        return imputed

# 5. Optimized feature-specific deep learning models
class OptimizedFeatureModels:
    def __init__(self, seed=42, column_names=None):
        self.seed = seed
        self.column_names = column_names
        self.models = {}
        self.scalers = {}
    
    def _build_feature_model(self, input_dim, feature_type='normal'):
        """Build an optimized deep learning model for one feature."""
        tf.random.set_seed(self.seed)
        
        initializer = initializers.GlorotUniform(seed=self.seed)
        
        if feature_type == 'h_feature':
            # Keep the current architecture for H(m)
            model = keras.Sequential([
                layers.Dense(64, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0005)),
                layers.BatchNormalization(),
                layers.Dropout(0.05, seed=self.seed),
                layers.Dense(32, activation='relu', kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0005)),
                layers.BatchNormalization(),
                layers.Dense(16, activation='relu', kernel_initializer=initializer),
                layers.Dense(8, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        elif feature_type == 'as_feature':
            # Use a deeper network with less regularization for As
            model = keras.Sequential([
                layers.Dense(96, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0003)),
                layers.BatchNormalization(),
                layers.Dropout(0.08, seed=self.seed),
                layers.Dense(48, activation='relu', kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.0003)),
                layers.BatchNormalization(),
                layers.Dense(24, activation='relu', kernel_initializer=initializer),
                layers.Dense(12, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        elif feature_type == 'theta_overfitted':
            # Use a small, strongly regularized network for θ
            model = keras.Sequential([
                layers.Dense(12, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.03)),
                layers.Dropout(0.5, seed=self.seed),
                layers.Dense(6, activation='relu', kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.03)),
                layers.Dense(1, kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.02))
            ])
        else:
            # Use the standard architecture for other features
            model = keras.Sequential([
                layers.Dense(32, activation='relu', input_shape=(input_dim,),
                            kernel_initializer=initializer,
                            kernel_regularizer=regularizers.l2(0.005)),
                layers.Dropout(0.2, seed=self.seed),
                layers.Dense(16, activation='relu', kernel_initializer=initializer),
                layers.Dropout(0.1, seed=self.seed+1),
                layers.Dense(8, activation='relu', kernel_initializer=initializer),
                layers.Dense(1, kernel_initializer=initializer)
            ])
        
        # Set the learning rate by feature
        if feature_type == 'h_feature':
            lr = 0.003
        elif feature_type == 'as_feature':
            lr = 0.004  # Increase the learning rate for As
        elif feature_type == 'theta_overfitted':
            lr = 0.0002  # Use a much lower learning rate for θ
        else:
            lr = 0.001
        
        model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='mse')
        return model
    
    def fit(self, data, mask):
        """Train a separate deep learning model for each feature."""
        print("Training optimized feature-specific deep learning models...")
        
        for col_idx in range(data.shape[1]):
            if self.column_names is not None:
                col_name = self.column_names[col_idx]
            else:
                col_name = f"feature_{col_idx}"
            
            obs_mask = mask[:, col_idx] == 1
            
            if np.sum(obs_mask) < 100:
                continue
            
            # Prepare the training data
            X_train = np.delete(data[obs_mask], col_idx, axis=1)
            y_train = data[obs_mask, col_idx]
            
            # Scale the data
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            X_train_scaled = scaler_x.fit_transform(X_train)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            
            # Set the seed
            tf.random.set_seed(self.seed + col_idx)
            np.random.seed(self.seed + col_idx)
            
            # Identify the feature
            if 'H' in col_name:
                feature_type = 'h_feature'
                patience = 40
                epochs = 350
            elif 'As' in col_name:
                feature_type = 'as_feature'
                patience = 50
                epochs = 400  # Increase the number of training epochs
            elif 'θ' in col_name:
                feature_type = 'theta_overfitted'
                patience = 8  # Reduce early-stopping patience
                epochs = 60  # Reduce the number of training epochs
            else:
                feature_type = 'normal'
                patience = 20
                epochs = 150
            
            # Build and train the model
            model = self._build_feature_model(X_train.shape[1], feature_type)
            
            early_stop = callbacks.EarlyStopping(monitor='loss', patience=patience, restore_best_weights=True)
            
            model.fit(
                X_train_scaled, y_train_scaled,
                epochs=epochs,
                batch_size=32,
                verbose=0,
                callbacks=[early_stop],
                shuffle=False
            )
            
            self.models[col_idx] = model
            self.scalers[col_idx] = {'x': scaler_x, 'y': scaler_y}
    
    def impute(self, data, mask):
        """Impute missing values."""
        imputed = data.copy()
        
        for col_idx, model in self.models.items():
            miss_mask = mask[:, col_idx] == 0
            if np.sum(miss_mask) == 0:
                continue
            
            X_miss = np.delete(imputed[miss_mask], col_idx, axis=1)
            X_miss_scaled = self.scalers[col_idx]['x'].transform(X_miss)
            pred_scaled = model.predict(X_miss_scaled, verbose=0).flatten()
            pred = self.scalers[col_idx]['y'].inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
            
            imputed[miss_mask, col_idx] = pred
        
        return imputed

# 6. Optimized ensemble without histogram gradient boosting
class OptimizedEnsemble:
    def __init__(self, data_dim, seed=42, column_names=None):
        self.data_dim = data_dim
        self.seed = seed
        self.column_names = column_names
        self.models = []
        self.weights = []
        self.feature_weights = {}
    
    def fit(self, data, mask, column_names):
        """Train the ensemble without HistGradientBoosting."""
        print("\nTraining the ensemble without HistGradientBoosting...")
        
        # 1. KNN
        print("1. KNN imputer...")
        knn = KNNImputer(n_neighbors=20, weights='distance')
        knn.fit(data)
        self.models.append(('knn', knn))
        
        # 2. GradientBoosting IterativeImputer
        print("2. GradientBoosting IterativeImputer...")
        gb_mice = IterativeImputer(
            estimator=GradientBoostingRegressor(
                n_estimators=200,  # Increase the setting to compensate for removing histogram gradient boosting
                max_depth=7,
                learning_rate=0.1,
                subsample=0.85,
                random_state=self.seed,
                validation_fraction=0.1,
                n_iter_no_change=5
            ),
            max_iter=35,  # Slightly increase the number of iterations
            random_state=self.seed,
            sample_posterior=False
        )
        gb_mice.fit(data)
        self.models.append(('gb_mice', gb_mice))
        
        # 3. RandomForest IterativeImputer
        print("3. RandomForest IterativeImputer...")
        rf_mice = IterativeImputer(
            estimator=RandomForestRegressor(
                n_estimators=150,  # Increase the number of trees to compensate for removing histogram gradient boosting
                max_depth=12,  # Slightly increase the depth
                min_samples_split=3,
                min_samples_leaf=2,
                max_features='sqrt',
                random_state=self.seed
            ),
            max_iter=30,  # Slightly increase the number of iterations
            random_state=self.seed,
            sample_posterior=False
        )
        rf_mice.fit(data)
        self.models.append(('rf_mice', rf_mice))
        
        # 4. ExtraTrees IterativeImputer
        print("4. ExtraTrees IterativeImputer...")
        et_mice = IterativeImputer(
            estimator=ExtraTreesRegressor(
                n_estimators=120,  # Slightly increase the number of trees
                max_depth=11,
                min_samples_split=3,
                max_features='sqrt',
                random_state=self.seed
            ),
            max_iter=30,  # Slightly increase the number of iterations
            random_state=self.seed,
            sample_posterior=False
        )
        et_mice.fit(data)
        self.models.append(('et_mice', et_mice))
        
        # 5. Feature-specific deep learning models
        print("5. Feature-specific deep learning models...")
        if hasattr(column_names, 'tolist'):
            column_names_list = column_names.tolist()
        else:
            column_names_list = list(column_names) if column_names is not None else None
        
        fs_model = OptimizedFeatureModels(seed=self.seed, column_names=column_names_list)
        fs_model.fit(data, mask)
        self.models.append(('feature_deep', fs_model))
        
        # 6. Optimized VAE
        print("6. Optimized VAE model...")
        vae = OptimizedVAE(data_dim=self.data_dim, latent_dim=14,
                          learning_rate=0.0004, seed=self.seed, 
                          column_names=column_names_list)
        
        # Initial imputation
        mean_imputer = SimpleImputer(strategy='mean')
        mean_imputer.fit(data)
        initial_imputed = mean_imputer.transform(data)
        data_filled = np.where(mask == 1, data, initial_imputed)
        
        vae.fit(data_filled, mask, epochs=900, batch_size=64)
        self.models.append(('vae', vae))
        
        # Redistribute the former histogram gradient boosting weight across models
        for i, col_name in enumerate(column_names):
            if 'V' in col_name:
                # V(m³): redistribute the weight across the other models
                self.feature_weights[i] = {
                    'knn': 0.12,  # +0.02
                    'gb_mice': 0.30,  # +0.10
                    'rf_mice': 0.25,  # +0.10
                    'et_mice': 0.13,  # +0.03
                    'feature_deep': 0.15,
                    'vae': 0.05
                }
            elif 'H' in col_name:
                # H(m): transfer most of the weight to gradient boosting
                self.feature_weights[i] = {
                    'knn': 0.10,
                    'gb_mice': 0.50,  # +0.20 from histogram gradient boosting
                    'rf_mice': 0.15,  # +0.05
                    'et_mice': 0.10,  # +0.05
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
            elif 'As' in col_name:
                # As(m²): transfer the weight to gradient boosting and random forest
                self.feature_weights[i] = {
                    'knn': 0.08,
                    'gb_mice': 0.40,  # +0.15
                    'rf_mice': 0.35,  # +0.15
                    'et_mice': 0.10,
                    'feature_deep': 0.05,
                    'vae': 0.02
                }
            elif 'θ' in col_name:
                # θ(°): give more weight to simpler models
                self.feature_weights[i] = {
                    'knn': 0.45,  # +0.05
                    'gb_mice': 0.35,  # +0.05
                    'rf_mice': 0.08,
                    'et_mice': 0.05,
                    'feature_deep': 0.04,
                    'vae': 0.03
                }
            else:
                # Wsm(m): distribute the weight evenly
                self.feature_weights[i] = {
                    'knn': 0.18,  # +0.03
                    'gb_mice': 0.32,  # +0.12
                    'rf_mice': 0.25,  # +0.10
                    'et_mice': 0.10,
                    'feature_deep': 0.10,
                    'vae': 0.05
                }
        
        print("Training complete for all six ensemble models")
    
    def impute(self, data, mask):
        """Predict using the ensemble."""
        n_samples, n_features = data.shape
        imputed = np.zeros_like(data)
        
        # Get predictions from all models
        all_predictions = {}
        for name, model in self.models:
            if name == 'vae':
                mean_imputer = SimpleImputer(strategy='mean')
                mean_imputer.fit(data)
                initial_imputed = mean_imputer.transform(data)
                data_filled = np.where(mask == 1, data, initial_imputed)
                pred = model.impute(data_filled, mask)
            elif name == 'feature_deep':
                pred = model.impute(data, mask)
            else:
                pred = model.transform(data)
            all_predictions[name] = pred
        
        # Take a weighted average for each feature
        for feature_idx in range(n_features):
            if feature_idx in self.feature_weights:
                weights = self.feature_weights[feature_idx]
                feature_pred = np.zeros(n_samples)
                
                for model_name, weight in weights.items():
                    if model_name in all_predictions:
                        feature_pred += all_predictions[model_name][:, feature_idx] * weight
                
                imputed[:, feature_idx] = feature_pred
            else:
                # Take the mean
                feature_pred = np.zeros(n_samples)
                n_models = len(all_predictions)
                for pred in all_predictions.values():
                    feature_pred += pred[:, feature_idx] / n_models
                imputed[:, feature_idx] = feature_pred
        
        # Preserve observed values
        imputed = data * mask + imputed * (1 - mask)
        
        return imputed

# 7. Evaluation
def deterministic_evaluate(model, data_array, mask, column_names, 
                          data_raw, scalers, transform_info):
    """Evaluate the model using deterministic cross-validation."""
    print("\n=== Deterministic evaluation in the original units ===")
    
    results = []
    all_observations = []
    
    original_index = data_raw.index
    
    for col_idx, col_name in enumerate(column_names):
        obs_mask = mask[:, col_idx] == 1
        obs_indices = np.where(obs_mask)[0]
        
        if len(obs_indices) < 100:
            continue
        
        # Use deterministic five-fold cross-validation
        kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
        
        all_true = []
        all_pred = []
        all_indices = []
        
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(obs_indices)):
            test_indices = obs_indices[test_idx]
            
            # Create the test mask
            test_mask = mask.copy()
            test_mask[test_indices, col_idx] = 0
            
            # Impute on the scaled data
            imputed_normalized = model.impute(data_array, test_mask)
            
            # Convert back to the original units
            imputed_original = inverse_transform_array(
                imputed_normalized, scalers, transform_info, 
                column_names, original_index
            )
            
            # Get the observed values in the original units
            true_values_original = data_raw.iloc[test_indices][col_name].values
            pred_values_original = imputed_original[col_name].iloc[test_indices].values
            
            all_true.extend(true_values_original)
            all_pred.extend(pred_values_original)
            all_indices.extend(test_indices)
            
            # Record each prediction
            for idx, true_val, pred_val in zip(test_indices, true_values_original, pred_values_original):
                all_observations.append({
                    'Row Index': idx,
                    'Feature': col_name,
                    'Fold': fold_idx,
                    'Observed Value': true_val,
                    'Predicted Value': pred_val,
                    'Absolute Error': abs(true_val - pred_val)
                })
        
        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        
        # Calculate metrics in the original units
        mae = mean_absolute_error(all_true, all_pred)
        rmse = np.sqrt(mean_squared_error(all_true, all_pred))
        
        # Calculate R²
        ss_res = np.sum((all_true - all_pred) ** 2)
        ss_tot = np.sum((all_true - np.mean(all_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        # Correlation coefficient
        if np.std(all_true) > 0 and np.std(all_pred) > 0:
            correlation = np.corrcoef(all_true, all_pred)[0, 1]
        else:
            correlation = 0
        
        results.append({
            'Feature': col_name,
            'R² (Original Scale)': r2,
            'Correlation': correlation,
            'MAE': mae,
            'RMSE': rmse,
            'Sample Count': len(all_true)
        })
        
        print(f"{col_name}: R²={r2:.4f}, correlation={correlation:.4f}, MAE={mae:.4f}")
    
    return pd.DataFrame(results), pd.DataFrame(all_observations)

# 8. Main function
def main():
    """Run the workflow."""
    print("=" * 80)
    print("Optimized VAE imputation without HistGradientBoosting")
    print("=" * 80)
    
    try:
        # Reset deterministic mode
        set_deterministic_mode()
        
        # 1. Load the data
        print("\n1. Loading data...")
        data_raw, original_data = load_and_preprocess_data()
        
        # Keep the original data for evaluation
        data_raw_copy = data_raw.copy()
        
        # 2. Treat outliers
        print("\n2. Treating outliers...")
        data_clean, transform_info = optimized_outlier_handling(data_raw)
        
        # 3. Scale the data
        print("\n3. Scaling the data...")
        data_normalized, scalers = optimized_normalization(data_clean)
        
        # Create the mask
        mask = (~data_normalized.isnull()).astype(np.float32).values
        data_array = data_normalized.fillna(0.5).values.astype(np.float32)
        
        print(f"\nData shape: {data_array.shape}")
        print(f"Missing rate: {(1 - mask.mean())*100:.2f}%")
        
        # 4. Train the optimized ensemble
        print("\n4. Training the optimized ensemble...")
        model = OptimizedEnsemble(data_dim=data_array.shape[1], seed=SEED, 
                                 column_names=data_normalized.columns)
        model.fit(data_array, mask, data_normalized.columns)
        
        # 5. Evaluate
        print("\n5. Evaluating the model...")
        eval_results, observations_df = deterministic_evaluate(
            model, data_array, mask, data_normalized.columns, 
            data_raw_copy, scalers, transform_info
        )
        
        # 6. Final imputation
        print("\n6. Generating the final imputed data...")
        final_imputed_normalized = model.impute(data_array, mask)
        
        # Convert back to the original units
        final_imputed_original = inverse_transform_array(
            final_imputed_normalized, scalers, transform_info, 
            data_normalized.columns, data_raw_copy.index
        )
        
        # Assemble the completed data
        final_result = data_raw_copy.astype(np.float64)
        for col in final_result.columns:
            missing_mask = data_raw_copy[col].isna()
            final_result.loc[missing_mask, col] = final_imputed_original.loc[missing_mask, col]
        
        # 7. Report the results
        print("\n" + "=" * 80)
        print("Model performance summary")
        print("=" * 80)
        
        for _, row in eval_results.iterrows():
            print(f"\n{row['Feature']}:")
            print(f"  R² (original scale): {row['R² (Original Scale)']:.4f}")
            print(f"  Correlation: {row['Correlation']:.4f}")
            print(f"  MAE: {row['MAE']:.4f}")
            print(f"  RMSE: {row['RMSE']:.4f}")
            print(f"  Evaluation samples: {row['Sample Count']}")
        
        print(f"\nMean R² (original scale): {eval_results['R² (Original Scale)'].mean():.4f}")
        print(f"Minimum R² (original scale): {eval_results['R² (Original Scale)'].min():.4f}")
        
        # Check whether the model meets the target
        min_r2 = eval_results['R² (Original Scale)'].min()
        if min_r2 >= 0.8:
            print("\n✅ All features meet the R² target of 0.8")
            print("✅ Model fit shows no clear signs of overfitting or underfitting")
        elif min_r2 >= 0.75:
            print(f"\n⚠️ Minimum R² is {min_r2:.4f}. Performance is good.")
        else:
            print(f"\n⚠️ Minimum R² is {min_r2:.4f}. Further optimization is needed.")
        
        # Check for potential overfitting
        high_r2_features = eval_results[eval_results['R² (Original Scale)'] > 0.94]['Feature'].tolist()
        if high_r2_features:
            print(f"\n⚠️ Features with possible mild overfitting: {', '.join(high_r2_features)}")
            print("   Stronger regularization is applied")
        
        # Check for potential underfitting
        low_r2_features = eval_results[eval_results['R² (Original Scale)'] < 0.75]['Feature'].tolist()
        if low_r2_features:
            print(f"\n⚠️ Features with possible underfitting: {', '.join(low_r2_features)}")
            print("   Consider increasing model capacity")
        
        # Save the results
        observations_df.to_csv("observed_vs_predicted_rainfall.csv", index=False, encoding='utf-8-sig')
        final_result.to_csv("imputed_rainfall.csv", index=False, encoding='utf-8-sig')
        eval_results.to_csv("evaluation_rainfall.csv", index=False, encoding='utf-8-sig')
        
        print("\nFiles saved:")
        print("1. observed_vs_predicted_rainfall.csv")
        print("2. imputed_rainfall.csv")
        print("3. evaluation_rainfall.csv")
        
        return final_result, eval_results, observations_df
        
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None

if __name__ == "__main__":
    print("\nStarting the workflow without HistGradientBoosting...")
    print("Main improvements:")
    print("1. Omit the IterativeImputer based on HistGradientBoosting")
    print("2. Adjust the gradient boosting and random forest settings")
    print("3. Redistribute model weights for each feature")
    print("4. Use six ensemble models instead of seven")
    print("5. Retain the other optimization settings")
    
    start_time = time.time()
    
    result, evaluation, observations = main()
    
    if result is not None:
        print(f"\n✅ Run complete!")
        print(f"Total time: {(time.time() - start_time)/60:.2f} minutes")
    else:
        print("\n❌ Run failed")
    
    gc.collect()

TensorFlow version: 2.21.0
Deterministic mode enabled
GPU disabled. Running on the CPU for reproducibility.


Starting the workflow without HistGradientBoosting...
Main improvements:
1. Omit the IterativeImputer based on HistGradientBoosting
2. Adjust the gradient boosting and random forest settings
3. Redistribute model weights for each feature
4. Use six ensemble models instead of seven
5. Retain the other optimization settings
Optimized VAE imputation without HistGradientBoosting
Deterministic mode enabled

1. Loading data...

Error: [Errno 2] No such file or directory: './Rainfall.csv'

✅ Run complete!
Total time: 0.00 minutes


Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10224\3604359494.py", line 1129, in main
    data_raw, original_data = load_and_preprocess_data()
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10224\3604359494.py", line 103, in load_and_preprocess_data
    data = pd.read_csv("./Rainfall.csv", low_memory=True)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\Anaconda3\envs\tensorflow\Lib\site-packages\pandas\io\parsers\readers.py", line

In [2]:
import pandas as pd

def add_L_column_to_results():
    """
    Add the L(m) column to the imputed results.
    """
    # 1. Read the source data to retrieve L(m)
    original_data = pd.read_csv("./Rainfall.csv", low_memory=True)
    
    # 2. Check whether L(m) is present
    if 'L(m)' not in original_data.columns:
        print("The source data does not contain L(m)")
        return
    
    # 3. Extract L(m)
    L_column = original_data['L(m)'].copy()
    print(f"Extracted L(m) with {len(L_column)} rows")
    
    # 4. Read the imputed results
    try:
        imputed_results = pd.read_csv("imputed_rainfall.csv")
        print(f"Imputed data shape: {imputed_results.shape}")
    except FileNotFoundError:
        print("Imputed results file not found. Run the imputation cell first.")
        return
    
    # 5. Check that the row counts match
    if len(imputed_results) != len(L_column):
        print(f"Warning: imputed results have {len(imputed_results)} rows, while L(m) has {len(L_column)} rows")
        # If row counts differ, align both tables to the shorter length
        min_rows = min(len(imputed_results), len(L_column))
        imputed_results = imputed_results.iloc[:min_rows]
        L_column = L_column.iloc[:min_rows]
    
    # 6. Add L(m) to the results
    imputed_results['L(m)'] = L_column.values
    
    # 7. Save the updated results
    output_filename = "imputed_rainfall_with_runout.csv"
    imputed_results.to_csv(output_filename, index=False)
    
    print(f"\nAdded L(m) to the imputed results.")
    print(f"Saved the new file as: {output_filename}")
    print(f"Final data shape: {imputed_results.shape}")
    print(f"Column names: {list(imputed_results.columns)}")
    
    # 8. Report summary statistics for L(m)
    print("\nL(m) summary statistics:")
    print(f"  Data type: {L_column.dtype}")
    print(f"  Unique values: {L_column.nunique()}")
    print(f"  Missing values: {L_column.isna().sum()}")
    if pd.api.types.is_numeric_dtype(L_column):
        print(f"  Minimum: {L_column.min()}")
        print(f"  Maximum: {L_column.max()}")
        print(f"  Mean: {L_column.mean():.2f}")
    else:
        print(f"  First five values: {L_column.head().tolist()}")
    
    return imputed_results

# Run the function
if __name__ == "__main__":
    result_with_L = add_L_column_to_results()

Extracted L(m) with 17486 rows
Imputed data shape: (17486, 5)

Added L(m) to the imputed results.
Saved the new file as: imputed_rainfall_with_runout.csv
Final data shape: (17486, 6)
Column names: ['As(m²）', 'H(m)', 'V(m³)', 'Wsm(m)', 'θ(°)', 'L(m)']

L(m) summary statistics:
  Data type: float64
  Unique values: 8135
  Missing values: 0
  Minimum: 0.42
  Maximum: 4590.0
  Mean: 62.20
